# DSFB Lattice / Phonon Reproducibility Notebook

This notebook rebuilds and runs the standalone Rust crate `crates/dsfb-lattice` from scratch, then loads the generated artifacts, displays the figures inline, and confirms the PDF and zip outputs.

## Why this notebook exists

The notebook exists to provide a reproducible empirical companion to a bounded subset of the paper *Deterministic Structural Inference in Solid-State Systems: A DSFB Engine for Crystal Lattices, Phonons, and Structural Forensics*. The scope is intentionally narrow: a deterministic fixed-end harmonic lattice, controlled perturbations, finite-dimensional spectral comparisons, residual construction, normalized residual metrics, baseline-derived envelope detectability, a canonical evaluation layer, and a transparent heuristic-bank retrieval step.

## What it demonstrates

For a toy 1D mass-spring chain, the crate constructs a dynamical matrix

$$
D = M^{-1/2} K M^{-1/2},
$$

computes its eigenpairs, interprets phonon-like frequencies as $\omega_i = \sqrt{\lambda_i}$ for non-negative eigenvalues, then introduces controlled perturbations to obtain

$$
D' = D + \Delta.
$$

The notebook then inspects finite-matrix spectral shifts, simulated observations, residuals, drift, and slew:

$$
r_k = y_{\mathrm{meas},k} - y_{\mathrm{pred},k}, \qquad
d_k = r_k - r_{k-1}, \qquad
s_k = d_k - d_{k-1}.
$$

An envelope is built from controlled nominal baseline variability, and a point-defect case is checked against that envelope for finite-time crossing. A grouped perturbation is compared against a localized defect through residual covariance structure. A softening sweep pushes a relevant eigenvalue toward zero and tracks the associated growth of residual-based indicators. The crate also exports canonical evaluation quantities for run-to-run comparability and applies a small heuristic bank that ranks candidate perturbation classes with explicit ambiguity signaling instead of forcing a unique label.

## What it does not claim

This notebook does **not** claim to validate the full paper, solve real-material phonon inference, establish universal detectability thresholds, provide universal defect identification, or establish universal heuristic-bank classification. It is a deterministic toy-model study designed to be inspectable and reproducible. The numerical inequality check for

$$
|\lambda_i' - \lambda_i| \leq \|\Delta\|_2
$$

is only an empirical illustration on the finite symmetric matrices generated here.

## How the Rust crate and notebook are connected

The Rust crate is the source of truth for the data generation. It creates timestamped run directories, computes spectra and time-series outputs, writes CSV / JSON artifacts, renders PNG figures, generates a PDF report, and writes a zip archive. The notebook does not replace that logic. Instead, it rebuilds the crate from scratch, executes the binary, loads the artifacts, displays the figures inline, inspects the canonical metrics and heuristic rankings, and performs a final reproducibility check on the PDF and zip outputs.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import zipfile

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

REPO_URL = os.environ.get("DSFB_LATTICE_REPO_URL", "https://github.com/infinityabundance/dsfb.git")

def resolve_repo() -> Path:
    cwd = Path.cwd()
    if (cwd / "crates" / "dsfb-lattice" / "Cargo.toml").exists():
        return cwd
    candidate = Path("/content/dsfb")
    if (candidate / "crates" / "dsfb-lattice" / "Cargo.toml").exists():
        return candidate
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(candidate)], check=True)
    return candidate

REPO_DIR = resolve_repo()
CRATE_DIR = REPO_DIR / "crates" / "dsfb-lattice"
OUTPUT_ROOT = CRATE_DIR / "output-dsfb-lattice"
os.chdir(REPO_DIR)

print("Repository:", REPO_DIR)
print("Crate:", CRATE_DIR)
print("Output root:", OUTPUT_ROOT)


In [ ]:
def ensure_rust_toolchain() -> None:
    cargo_path = shutil.which("cargo")
    cargo_home_bin = Path.home() / ".cargo" / "bin"
    if cargo_home_bin.exists():
        os.environ["PATH"] = f"{cargo_home_bin}:{os.environ['PATH']}"
        cargo_path = shutil.which("cargo")
    if cargo_path is None:
        subprocess.run(
            "curl https://sh.rustup.rs -sSf | sh -s -- -y",
            shell=True,
            check=True,
        )
        os.environ["PATH"] = f"{cargo_home_bin}:{os.environ['PATH']}"
    subprocess.run(["cargo", "--version"], check=True)

ensure_rust_toolchain()


In [ ]:
subprocess.run(["cargo", "clean", "--manifest-path", str(CRATE_DIR / "Cargo.toml")], check=True)

result = subprocess.run(
    [
        "cargo",
        "run",
        "--release",
        "--manifest-path",
        str(CRATE_DIR / "Cargo.toml"),
        "--",
        "--example",
        "all",
        "--output-root",
        str(OUTPUT_ROOT),
    ],
    check=True,
    text=True,
    capture_output=True,
)

print(result.stderr)
print(result.stdout)

def parse_marker(name: str) -> Path:
    for line in result.stdout.splitlines():
        if line.startswith(name + "="):
            return Path(line.split("=", 1)[1].strip())
    raise RuntimeError(f"Could not find {name} in cargo output")

RUN_DIR = parse_marker("RUN_DIRECTORY")
SUMMARY_PATH = parse_marker("SUMMARY_JSON")
REPORT_PDF = parse_marker("REPORT_PDF")
ZIP_PATH = parse_marker("ZIP_ARCHIVE")

summary = json.loads(SUMMARY_PATH.read_text())

print("Run directory:", RUN_DIR)
print("Summary JSON:", SUMMARY_PATH)
print("Report PDF:", REPORT_PDF)
print("Zip archive:", ZIP_PATH)


In [ ]:
experiments_df = pd.DataFrame(summary["experiments"])
metrics_df = pd.read_csv(RUN_DIR / "metrics.csv")
detectability = summary.get("detectability")
detectability_columns = [
    "first_crossing_step",
    "first_crossing_time",
    "signal_at_first_crossing",
    "envelope_at_first_crossing",
    "crossing_margin",
    "consecutive_crossing_step",
    "consecutive_crossing_time",
    "global_signal_peak",
    "global_signal_peak_time",
    "global_envelope_peak",
    "global_envelope_peak_time",
]
detectability_df = (
    pd.DataFrame([detectability])[detectability_columns]
    if detectability is not None
    else pd.DataFrame(columns=detectability_columns)
)
softening_df = pd.read_csv(RUN_DIR / "softening_sweep.csv")
canonical_df = pd.read_csv(RUN_DIR / "canonical_metrics.csv")
pressure_test_df = pd.read_csv(RUN_DIR / "pressure_test_summary.csv")
heuristic_df = pd.read_csv(RUN_DIR / "heuristic_rankings.csv")
heuristic_top_df = heuristic_df[heuristic_df["candidate_rank"] == 1]

display(Markdown("### Experiment Summary"))
display(
    experiments_df[
        [
            "name",
            "delta_norm_2",
            "max_abs_shift",
            "bound_satisfied",
            "max_residual_norm",
            "max_drift_norm",
            "max_slew_norm",
            "covariance_offdiag_energy",
        ]
    ]
)

display(Markdown("### Detectability Summary"))
display(Markdown("Pointwise detectability uses the same-time condition `||r(t)|| > E(t)`. The table below separates global peaks over the full trajectory from the values observed at the first crossing time."))
display(detectability_df)

display(Markdown("### Softening Sweep (tail)"))
display(softening_df.tail())

display(Markdown("### Canonical Evaluation Quantities"))
display(Markdown("These rows are the synthetic benchmark comparison backbone exported by the Rust crate. Additional rows may exist elsewhere, but the canonical quantities are the intended run-to-run comparison layer."))
display(canonical_df)

display(Markdown("### Pressure-Test Comparison Table"))
display(pressure_test_df)

display(Markdown("### Heuristic-Bank Top Matches"))
display(Markdown("The heuristic bank is a constrained retrieval mechanism over compact descriptors. It ranks admissible candidates and can mark near-tied results as descriptor-space ambiguity."))
display(heuristic_top_df[["observed_case", "top_match", "top_distance", "ambiguity_flag", "ambiguity_gap"]])

display(Markdown("### Metrics CSV (selected rows)"))
display(metrics_df[metrics_df["experiment"].isin(["baseline", "detectability", "softening"])].head(24))


In [ ]:
figure_paths = [Path(path) for path in summary["figures"]]
for figure_path in figure_paths:
    display(Markdown(f"### `{figure_path.name}`"))
    image = mpimg.imread(figure_path)
    plt.figure(figsize=(12, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.show()


In [ ]:
report_md = Path(summary["report_markdown"])

if not REPORT_PDF.exists():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "pandoc", "texlive-xetex"], check=True)
    subprocess.run(["pandoc", str(report_md), "-o", str(REPORT_PDF)], check=True)

if not ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for file_path in sorted(RUN_DIR.rglob("*")):
            if file_path.is_file():
                archive.write(file_path, arcname=str(file_path.relative_to(RUN_DIR.parent)))

assert REPORT_PDF.exists(), "report.pdf was not generated"
assert ZIP_PATH.exists(), "zip archive was not generated"

print("Confirmed report PDF:", REPORT_PDF)
print("Confirmed zip archive:", ZIP_PATH)


In [ ]:
point = experiments_df.loc[experiments_df["name"] == "point_defect"].iloc[0]
group = experiments_df.loc[experiments_df["name"] == "group_mode_cluster"].iloc[0]
softening = summary["softening"]
detectability = summary.get("detectability")

if detectability is None:
    pointwise_text = "- No detectability summary was produced for this run."
    peak_context_text = ""
elif detectability["first_crossing_step"] is None:
    pointwise_text = "- No pointwise envelope crossing was observed, so the first-crossing values remain null in `summary.json`."
    peak_context_text = (
        f"- The global signal peak was {detectability['global_signal_peak']:.4f} at time {detectability['global_signal_peak_time']:.2f}, while the global envelope peak was {detectability['global_envelope_peak']:.4f} at time {detectability['global_envelope_peak_time']:.2f}. Global peaks need not occur at the same time and do not by themselves determine detectability."
    )
else:
    sustained_suffix = ""
    if detectability["consecutive_crossing_step"] is not None:
        sustained_suffix = (
            f" The configured sustained-crossing rule first triggered at step {detectability['consecutive_crossing_step']} (time {detectability['consecutive_crossing_time']:.2f})."
        )
    pointwise_text = (
        f"- The first pointwise envelope crossing occurred at step {detectability['first_crossing_step']} (time {detectability['first_crossing_time']:.2f}) because the residual norm there was {detectability['signal_at_first_crossing']:.4f} while the envelope at that same time was {detectability['envelope_at_first_crossing']:.4f}, giving a crossing margin of {detectability['crossing_margin']:.4f}.{sustained_suffix}"
    )
    peak_context_text = (
        f"- The global signal peak was {detectability['global_signal_peak']:.4f} at time {detectability['global_signal_peak_time']:.2f}, while the global envelope peak was {detectability['global_envelope_peak']:.4f} at time {detectability['global_envelope_peak_time']:.2f}. These global peaks need not occur at the same time and do not by themselves determine detectability."
    )

display(
    Markdown(
        f"""
### Concise Findings Summary

This notebook rebuilt the Rust crate from scratch, executed the deterministic demo pipeline, and wrote a fresh timestamped run under `{RUN_DIR}`.

- For the finite symmetric toy operators generated here, the reported sorted spectral shifts remained below the computed `||Delta||_2` values.
{pointwise_text}
{peak_context_text}
- The grouped perturbation produced larger off-diagonal residual covariance energy (`{group['covariance_offdiag_energy']:.4f}`) than the localized point defect (`{point['covariance_offdiag_energy']:.4f}`), which is consistent with broader correlated modal impact in this toy setting.
- The softening sweep reduced the smallest eigenvalue to `{softening['softest_smallest_eigenvalue']:.4f}` at spring scale `{softening['softest_scale']:.2f}` while residual, drift, and slew indicators increased.

These are bounded empirical demonstrations on a deterministic harmonic toy model. They do not establish universal detectability, universal phonon inference, or general transition prediction across materials.
"""
    )
)
